# Metric Guide

TMAP supports four input paths. Choose the one that matches your data.


## Summary

| Your data | Metric | Backend | Good for |
|-----------|--------|---------|----------|
| Binary matrix | `jaccard` | USearch HNSW (Jaccard on raw bits) | Molecular fingerprints and other 0/1 features |
| Sparse matrix (CSR) | `jaccard` | MinHash + LSHForest | Large sparse binary data (e.g. single-cell) |
| List of sets/strings | `jaccard` | MinHash + LSHForest | Variable-length token collections |
| Dense float matrix | `cosine` | USearch HNSW | Embeddings where direction matters |
| Dense float matrix | `euclidean` | USearch HNSW | Embeddings where magnitude matters |
| Distance matrix | `precomputed` | Direct graph construction | Distances you already computed |

The backend is selected automatically based on the input type. Binary matrices get USearch (high recall, low memory). Sparse matrices and variable-length inputs get MinHash+LSH.

## 1. Jaccard For Binary Fingerprints

Binary matrices are routed to USearch automatically. USearch builds an HNSW graph using native bitwise Jaccard distance — no MinHash encoding needed. This gives ~98% recall vs ~1% for MinHash+LSH on sparse fingerprints.

In [1]:
import numpy as np

from tmap import TMAP

rng = np.random.default_rng(42)
X_binary = rng.integers(0, 2, size=(2000, 2048), dtype=np.uint8)

jaccard_model = TMAP(metric="jaccard", n_neighbors=20, kc=50, seed=42).fit(X_binary)
print(jaccard_model.embedding_.shape)


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


(2000, 2)


## 2. Cosine For Dense Embeddings


In [3]:
%pip install usearch
X_dense = rng.normal(size=(2000, 128)).astype(np.float32)

cosine_model = TMAP(metric="cosine", n_neighbors=20, seed=42).fit(X_dense)
print(cosine_model.embedding_.shape)


  Using cached usearch-2.23.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (33 kB)
  Using cached simsimd-6.5.16-cp312-cp312-macosx_11_0_arm64.whl.metadata (70 kB)
Using cached usearch-2.23.0-cp312-cp312-macosx_11_0_arm64.whl (401 kB)
Using cached simsimd-6.5.16-cp312-cp312-macosx_11_0_arm64.whl (94 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [usearch]
Note: you may need to restart the kernel to use updated packages.
(2000, 2)


## 3. Euclidean For Dense Embeddings


In [4]:
euclidean_model = TMAP(metric="euclidean", n_neighbors=20, seed=42).fit(X_dense)
print(euclidean_model.embedding_.shape)


(2000, 2)


## 4. Precomputed Distances


In [5]:
dist = np.linalg.norm(X_dense[:, None, :] - X_dense[None, :, :], axis=2).astype(np.float32)

precomputed_model = TMAP(metric="precomputed", n_neighbors=20, seed=42).fit(dist)
print(precomputed_model.embedding_.shape)


(2000, 2)


## Rule Of Thumb

- use `jaccard` with a binary matrix for chemistry fingerprints (auto-uses USearch)
- use `jaccard` with a sparse matrix for large single-cell data (auto-uses MinHash+LSH)
- use `cosine` for learned embeddings
- use `euclidean` when vector magnitude matters
- use `precomputed` when another tool already gave you distances

See `docs/usearch_backend.md` for detailed benchmarks comparing USearch vs MinHash+LSH.